In [ ]:
pip install -U datasets


In [ ]:
!pip install --upgrade datasets fsspec


  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)


In [ ]:
!pip install konlpy JPype1


In [ ]:
# 욕설 사전 로드
def load_bad_words(path):
    with open(path, encoding='utf-8') as f:
        return set(w.strip() for w in f if w.strip())

bad_words = load_bad_words('extended_bad_words.txt')

In [ ]:
from konlpy.tag import Okt

# 형태소 분석기 초기화 (한 번만)
okt = Okt()

# 핵심 처리 함수
def contains_bad_word_loose(text: str, bad_words: set) -> int:
    # 1차 필터링: 띄어쓰기로 구분된 욕설 단어 확인
    words = text.split()  # 공백으로 단어 분리
    for word in words:
        if word.lower() in bad_words:  # 욕설 목록에 정확히 일치하면
            return 1  # 욕설이 발견되면 1 반환

    # 2차 필터링: 형태소 분석을 통한 욕설 체크
    tokens = okt.pos(text, norm=True, stem=True)  # 형태소 분석

    for idx, (tok, tag) in enumerate(tokens):
        t = tok.lower()  # 소문자로 변환하여 일관된 비교
        for bw in bad_words:
            if not bw:
                continue

            # 1. "개"에 대한 예외 처리: "개" + 욕설, 욕설 + "개"
            if t == '개':
                if idx + 1 < len(tokens) and tokens[idx + 1][0].lower() in bad_words:  # "개" 뒤에 욕설
                    return 1  # 욕설 발견
                if idx - 1 >= 0 and tokens[idx - 1][0].lower() in bad_words:  # 욕설 + "개"
                    return 1  # 욕설 발견
                continue  # "개"는 단독일 때 욕설 아님

            # 2. "년"에 대한 예외 처리: 숫자와 함께 있을 때는 욕설이 아님
            if t == '년':
                if idx - 1 >= 0 and tokens[idx - 1][1] == 'Number':  # "3년", "5년"은 욕설 아님
                    continue
                return 1  # "년"이 욕설 + 결합되면 욕설로 감지

            # 3. "새끼"에 대한 예외 처리: "새끼" 뒤에 명사인 경우는 욕설 아님
            if t == '새끼':
                if idx + 1 < len(tokens) and tokens[idx + 1][1] == 'Noun' and tokens[idx + 1][0].lower() not in bad_words:
                    continue  # 일반 명사라면 예외 처리
                return 1  # 욕설 + "새끼", "새끼" + 욕설은 욕설로 처리

            # 4. "자식"에 대한 예외 처리: "자식"은 욕설과 결합될 때만 욕설로 처리
            if t == '자식':
                if idx - 1 >= 0 and (tokens[idx - 1][0].lower() in bad_words or tokens[idx - 1][0].lower() == '개'):
                    return 1  # 욕설 + "자식", "개" + "자식"은 욕설로 처리
                if idx + 1 < len(tokens) and tokens[idx + 1][0].lower() in bad_words:
                    return 1  # "자식" + 욕설은 욕설로 처리
                continue  # "자식"은 단독으로 욕설이 아님

            # 5. 정확히 일치하는 욕설 찾기
            elif t == bw:
                return 1  # 정확히 일치하면 욕설로 추가

            # 6. 접두사 일치하는 욕설 찾기
            elif t.startswith(bw):
                suffix = t[len(bw):]  # 접두사 이후 남은 부분
                sp = okt.pos(suffix, norm=True, stem=True)  # 접미사 처리
                if len(sp) == 1 and sp[0][1] == 'Noun' and sp[0][0].lower() not in bad_words:
                    continue  # 뒤에 명사가 오고 그 명사가 욕설이 아니면 스킵
                return 1  # 접두사가 욕설인 경우 추가

    return 0  # 욕설이 없으면 0 반환


In [ ]:
# ──────────────────────────────────────────────────────
# 사용 예시
if __name__ == '__main__':
    bad_words = load_bad_words('extended_bad_words.txt')  # 사전 로드
    tests = [
        "개새끼",
        "너 진짜 개새끼구나?",
        "새끼손가락걸고 약속하자.",
        "새끼강아지는 귀엽네",
        "병신이지 너?",
        "학교나 가자",
        '개같은 집에서 살고있냐?',
        '시바견 귀여워~',
        '3년이 흘렀다',
        '무식한 년아 이것도 몰라?',
        '씨발새끼야',
        '미친년 아님?',
        '야이 시발이 개새끼야'
    ]
    for t in tests:
        detected = contains_bad_word_loose(t, bad_words)
        print(f"{t:<30} → {detected}")

개새끼                            → 1
너 진짜 개새끼구나?                    → 1
새끼손가락걸고 약속하자.                  → 0
새끼강아지는 귀엽네                     → 0
병신이지 너?                        → 1
학교나 가자                         → 0
개같은 집에서 살고있냐?                  → 1
시바견 귀여워~                       → 0
3년이 흘렀다                        → 0
무식한 년아 이것도 몰라?                 → 1
씨발새끼야                          → 1
미친년 아님?                        → 1
야이 시발이 개새끼야                    → 1


In [ ]:
def contains_bad_word_loose(text: str, bad_words: set) -> bool:
    # 1차 필터링: 띄어쓰기로 구분된 욕설 단어 확인
    words = text.split()  # 공백으로 단어 분리
    for word in words:
        if word.lower() in bad_words:  # 욕설 목록에 정확히 일치하면 추가
            return True  # 욕설 발견 시 True 반환

    # 2차 필터링: 형태소 분석을 통한 욕설 체크
    tokens = okt.pos(text, norm=True, stem=True)  # 형태소 분석

    for idx, (tok, tag) in enumerate(tokens):
        t = tok.lower()  # 소문자로 변환하여 일관된 비교
        for bw in bad_words:
            if not bw:
                continue

            # 예외 처리: "개", "년", "새끼", "자식" 등
            if t == bw:
                return True  # 정확히 일치하는 욕설 찾으면 True 반환

            # 예외 처리 로직: "개", "년", "새끼", "자식" 등
            if t == '개':
                if idx + 1 < len(tokens) and tokens[idx + 1][0].lower() in bad_words:  # "개" 뒤에 욕설
                    return True
                if idx - 1 >= 0 and tokens[idx - 1][0].lower() in bad_words:  # 욕설 + "개"
                    return True
                continue

            if t == '년':
                if idx - 1 >= 0 and tokens[idx - 1][1] == 'Number':  # "3년", "5년"은 욕설 아님
                    continue
                return True

            if t == '새끼':
                if idx + 1 < len(tokens) and tokens[idx + 1][1] == 'Noun' and tokens[idx + 1][0].lower() not in bad_words:
                    continue
                return True

            if t == '자식':
                if idx - 1 >= 0 and (tokens[idx - 1][0].lower() in bad_words or tokens[idx - 1][0].lower() == '개'):
                    return True
                if idx + 1 < len(tokens) and tokens[idx + 1][0].lower() in bad_words:
                    return True
                continue

    return False  # 욕설이 없으면 False 반환


EXTENDED_FILE = "extended_bad_words.txt"
  # 결과 출력
# 예시 텍스트
transcript = "씹쌔끼야"

# 욕설 탐지 및 경고
swears = contains_bad_word_loose(transcript, load_bad_words(EXTENDED_FILE))
if swears:
    print(f"⚠️ 욕설 발견됨")
else:
    print("✅ 욕설 없음.")

⚠️ 욕설 발견됨


In [ ]:
# ──────────────────────────────────────────────────────
# 사용 예시
if __name__ == '__main__':
    bad_words = load_bad_words('extended_bad_words.txt')  # 사전 로드
    tests = [
        "개새끼",
        "너 진짜 개새끼구나?",
        "새끼손가락걸고 약속하자.",
        "새끼강아지는 귀엽네",
        "병신이지 너?",
        "학교나 가자",
        '개같은 집에서 살고있냐?',
        '시바견 귀여워~',
        '3년이 흘렀다',
        '무식한 년아 이것도 몰라?',
        '씨발새끼야',
        '미친년 아님?',
        '야이 시발이 개새끼야'
    ]
    for t in tests:
        detected = contains_bad_word_loose(t, bad_words)
        print(f"{t:<30} → {detected}")

개새끼                            → True
너 진짜 개새끼구나?                    → True
새끼손가락걸고 약속하자.                  → False
새끼강아지는 귀엽네                     → True
병신이지 너?                        → True
학교나 가자                         → False
개같은 집에서 살고있냐?                  → True
시바견 귀여워~                       → False
3년이 흘렀다                        → False
무식한 년아 이것도 몰라?                 → True
씨발새끼야                          → True
미친년 아님?                        → True
야이 시발이 개새끼야                    → True
